# weight-decay-decoupled — ex1: implement one AdamW step with decoupled weight decay

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `weight-decay-decoupled`. Running the final beacon cell reports progress against the `Optimizer: decoupled weight decay (AdamW)` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: decoupled weight decay (AdamW)` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`weight-decay-decoupled`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "weight-decay-decoupled"
DD_SUBTOPIC = "Optimizer: decoupled weight decay (AdamW)"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Optimizer: decoupled weight decay (AdamW) — quick refresher

`Adam` and `AdamW` differ in ONE line. Adam folds weight decay into the gradient *before* the adaptive moment estimates run:

```python
# Adam (coupled): wd added to grad, gets m/v statistics applied to it
g = p.grad + wd * p
m = beta1*m + (1-beta1)*g
v = beta2*v + (1-beta2)*g*g
p -= lr * m / (sqrt(v) + eps)
```

AdamW *decouples* the decay — it's applied to the parameters directly, as a SEPARATE update from the Adam moment step, untouched by the moment statistics. `torch.optim.AdamW` applies the decay BEFORE the Adam step:

```python
# AdamW (decoupled): two separate updates
p *= (1 - lr * wd)              # decoupled wd step (== p -= lr*wd*p)
m = beta1*m + (1-beta1)*p.grad
v = beta2*v + (1-beta2)*p.grad*p.grad
p -= lr * m / (sqrt(v) + eps)   # Adam step
```

**Why this matters.** In Adam, the decay term flows through `sqrt(v)` normalization, so weights with large gradient variance effectively get LESS decay — exactly backwards of what L2 regularization is supposed to do (Loshchilov & Hutter 2017). AdamW restores the original L2-reg meaning.

**The combined-step form.** A common compact rewrite is `p -= lr * (m/(sqrt(v)+eps) + wd * p)`. Mathematically equivalent to the two-line version above; you'll see both in the wild.

### Exercise 1 — implement one AdamW step with decoupled weight decay

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the decoupled weight-decay update `p -= lr * wd * p` AS A SEPARATE STEP from the Adam moment update, distinguishing AdamW from Adam.
> Keywords: adamw, weight-decay, decoupled, optimizer-step
> ```

**KCs targeted:** `adamw-decoupled-decay-formula`, `adam-vs-adamw-difference`

Implement `ex1_adamw_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step)`. ONE AdamW update step on a single parameter tensor `p` (in place).

Inputs:
- `p`: parameter `Tensor` (modified in place).
- `grad`: gradient `Tensor`, same shape as `p`.
- `m`, `v`: 1st/2nd moment running estimates (same shape as `p`, modified in place).
- `lr`, `beta1`, `beta2`, `eps`, `wd`: scalar floats.
- `step`: int >= 1, the current step count (for bias correction).

Algorithm — the decoupled form (matches `torch.optim.AdamW`):
1. **Decoupled weight-decay step** (applied directly to the parameter, NOT mixed into the gradient — this is what makes it AdamW instead of Adam):
   `p <- p * (1 - lr * wd)`   (i.e. `p -= lr * wd * p`)
2. **Moment update** (Adam, unchanged):
   `m <- beta1*m + (1-beta1)*grad`
   `v <- beta2*v + (1-beta2)*grad*grad`
3. **Bias correction**:
   `m_hat = m / (1 - beta1^step)`
   `v_hat = v / (1 - beta2^step)`
4. **Adam step** (no wd inside):
   `p <- p - lr * m_hat / (sqrt(v_hat) + eps)`

Return nothing — `p`, `m`, `v` are updated in place. The test compares against a hand-computed reference and against `torch.optim.AdamW` itself.

Note on ordering: PyTorch's `torch.optim.AdamW` applies the decay BEFORE the Adam step (as above). The original Loshchilov & Hutter paper allowed either order; for small `lr` they are numerically indistinguishable, but the test uses PyTorch's reference so we match its order exactly.

In [ ]:
def ex1_adamw_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    """One in-place AdamW step on (p, m, v)."""
    raise NotImplementedError()


def _test_ex1():
    # === Reference: hand-computed AdamW step on a single scalar ===
    # p_0 = 1.0, grad = 0.5, m=v=0, lr=0.1, b1=0.9, b2=0.999, eps=1e-8, wd=0.01
    # step=1
    # 1. Decoupled decay (pre-Adam): p = 1.0 * (1 - 0.1 * 0.01) = 0.999
    # 2. m = 0.9*0 + 0.1*0.5 = 0.05
    # 3. v = 0.999*0 + 0.001*0.25 = 0.00025
    # 4. m_hat = 0.05 / (1 - 0.9) = 0.5
    # 5. v_hat = 0.00025 / (1 - 0.999) = 0.25
    # 6. Adam step: p = 0.999 - 0.1 * 0.5 / (sqrt(0.25) + 1e-8) ~= 0.999 - 0.1 = 0.899
    p = t.tensor([1.0])
    grad = t.tensor([0.5])
    m = t.tensor([0.0])
    v = t.tensor([0.0])
    ex1_adamw_step(p, grad, m, v, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, wd=0.01, step=1)
    assert abs(p.item() - 0.899) < 1e-4, f'expected p~=0.899, got {p.item()}'
    assert abs(m.item() - 0.05) < 1e-7, f'expected m=0.05, got {m.item()}'
    assert abs(v.item() - 0.00025) < 1e-9, f'expected v=0.00025, got {v.item()}'

    # === Cross-check against torch.optim.AdamW on a small linear model ===
    t.manual_seed(0)
    p_ref = t.nn.Parameter(t.randn(5))
    p_ours = p_ref.detach().clone().requires_grad_(False)
    g = t.randn(5)
    p_ref.grad = g.clone()

    lr, b1, b2, eps, wd = 1e-2, 0.9, 0.999, 1e-8, 0.05
    opt = t.optim.AdamW([p_ref], lr=lr, betas=(b1, b2), eps=eps, weight_decay=wd)
    opt.step()

    m = t.zeros_like(p_ours)
    v = t.zeros_like(p_ours)
    ex1_adamw_step(p_ours, g.clone(), m, v, lr=lr, beta1=b1, beta2=b2, eps=eps, wd=wd, step=1)

    assert t.allclose(p_ours, p_ref.detach(), atol=1e-6), (
        f'AdamW step mismatch:\n  ours = {p_ours}\n  ref  = {p_ref.detach()}'
    )

    # === Multi-step cross-check ===
    t.manual_seed(1)
    p_ref = t.nn.Parameter(t.randn(4))
    p_ours = p_ref.detach().clone().requires_grad_(False)
    m_ours = t.zeros_like(p_ours)
    v_ours = t.zeros_like(p_ours)
    opt = t.optim.AdamW([p_ref], lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.1)

    for step in range(1, 6):
        grad = t.randn(4, generator=t.Generator().manual_seed(step))
        p_ref.grad = grad.clone()
        opt.step()
        ex1_adamw_step(p_ours, grad.clone(), m_ours, v_ours,
                       lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8,
                       wd=0.1, step=step)

    assert t.allclose(p_ours, p_ref.detach(), atol=1e-5), (
        f'multi-step AdamW mismatch:\n  ours = {p_ours}\n  ref  = {p_ref.detach()}'
    )

    # === Compare to Adam (COUPLED) to confirm AdamW is different ===
    # Same hparams but with torch.optim.Adam, weight_decay=0.1, on the same grad.
    # AdamW result should NOT equal Adam result (they differ exactly because of
    # the decoupled-vs-coupled wd treatment).
    t.manual_seed(2)
    p_adam = t.nn.Parameter(t.randn(3))
    p_adamw = p_adam.detach().clone()
    g = t.randn(3)
    p_adam.grad = g.clone()

    opt_adam = t.optim.Adam([p_adam], lr=1e-2, weight_decay=0.1)
    opt_adam.step()

    m = t.zeros_like(p_adamw)
    v = t.zeros_like(p_adamw)
    ex1_adamw_step(p_adamw, g.clone(), m, v,
                   lr=1e-2, beta1=0.9, beta2=0.999, eps=1e-8, wd=0.1, step=1)

    diff = (p_adam.detach() - p_adamw).abs().max().item()
    assert diff > 1e-5, (
        f'AdamW and Adam(wd=0.1) should differ; diff={diff} — '
        f'did you fold wd into the gradient (Adam-style) instead of decoupling?'
    )

    # === wd=0 makes AdamW collapse to Adam ===
    t.manual_seed(3)
    p_adam = t.nn.Parameter(t.randn(3))
    p_adamw = p_adam.detach().clone()
    g = t.randn(3)
    p_adam.grad = g.clone()

    opt_adam = t.optim.Adam([p_adam], lr=1e-2, weight_decay=0.0)
    opt_adam.step()

    m = t.zeros_like(p_adamw)
    v = t.zeros_like(p_adamw)
    ex1_adamw_step(p_adamw, g.clone(), m, v,
                   lr=1e-2, beta1=0.9, beta2=0.999, eps=1e-8, wd=0.0, step=1)

    assert t.allclose(p_adam.detach(), p_adamw, atol=1e-6), (
        f'with wd=0 AdamW must equal Adam, got diff '
        f'{(p_adam.detach()-p_adamw).abs().max().item()}'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_adamw_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    # 1. Decoupled weight-decay step FIRST (matches torch.optim.AdamW):
    #    p <- p * (1 - lr * wd)   ==   p -= lr * wd * p
    p.mul_(1 - lr * wd)
    # 2. Moment update (Adam, unchanged)
    m.mul_(beta1).add_(grad, alpha=1 - beta1)
    v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
    # 3. Bias correction
    m_hat = m / (1 - beta1 ** step)
    v_hat = v / (1 - beta2 ** step)
    # 4. Adam step (no wd inside)
    p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)
```

**The ONE line that makes it AdamW.** Step 4 — `p -= lr * wd * p` — is the entire difference from Adam. Comment it out and you've reimplemented Adam (no decay). Move it into the gradient (`grad = grad + wd * p` before the moment update) and you've implemented coupled Adam-with-wd, which is what `torch.optim.Adam(weight_decay=...)` actually does.

**`addcdiv_(t1, t2, value=-lr)`** is the in-place form of `p += -lr * t1 / t2`. The PyTorch source uses this for the Adam step because it avoids a temporary tensor.

**Bias correction matters most at small `step`.** At `step=1`, `1 - beta1**1 = 0.1` so `m_hat = 10*m`. Without this, the first few steps would be tiny because `m` is biased toward zero (the EMA hasn't warmed up). By step ~1000 the correction factor is essentially 1.

**`p.mul_(1 - lr*wd)` is the pre-Adam decay.** That matches `torch.optim.AdamW`'s actual implementation in v1.13+ — decay first, then the Adam step on the now-decayed `p`. The original 2017 paper sketched both orderings; for small `lr` they are numerically indistinguishable, but PyTorch picked the pre-step form.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()